# Chemical space 반복 탐색

**한 번에 두 가지만 결정합니다.**

| 행동 | 화합물 집합 | 배치 기준 |
|---|---|---|
| 영역 선택 | 줄어듦 | 그대로 |
| 도메인 전환 | 그대로 | 바뀜 |

두 경우 모두 **선택된 화합물만으로 다시 임베딩·군집화**합니다. 전체 지도를 확대하는 것이 아니라서, 물성에서 붙어 있던 화합물이 질환 관점에서는 여러 군집으로 갈라집니다.

모든 단계는 트리의 노드로 남습니다. 이전 노드로 돌아가 다른 영역을 고르면 기존 가지를 지우지 않고 형제 가지가 생깁니다.

In [ ]:
import logging

import chemspace

logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("numba").setLevel(logging.WARNING)

ds = chemspace.load()  # data/cache 의 최신 ChEMBL 스냅샷
ds

## 도메인별 데이터 커버리지를 먼저 봅니다

ChEMBL은 도메인마다 커버리지가 크게 다릅니다. 물성·모이어티는 계산으로 전부 채워지지만, 질환과 표적은 해당 데이터가 있는 화합물에만 존재합니다. 데이터가 없는 화합물은 그 도메인에서 **원점이 아니라 '미지'** 이므로, 임베딩 적합에서 빼고 지도 아래쪽 띠에 따로 배치합니다.

In [ ]:
ds.coverage()

In [ ]:
ds.compounds["modality"].value_counts()

## 1단계: 전체를 물성 도메인으로 배치

In [ ]:
session = chemspace.ExplorationSession(ds)
root = session.root("physchem", note="전체 집합")
print(session.breadcrumb(root.node_id))

## 드래그로 탐색

1. 올가미(lasso)로 영역을 드래그 → **‘선택 영역으로 좁히기’**
2. 도메인 드롭다운을 바꾸고 → **‘도메인 전환’**
3. 반복

`단계` 드롭다운으로 아무 노드로나 돌아갈 수 있고, 거기서 다시 선택하면 분기가 생깁니다.

In [ ]:
explorer = chemspace.SpaceExplorer(session)
explorer.show()

## 같은 일을 코드로

뷰어는 세션에 상태를 두지 않습니다. 드래그로 하는 모든 동작은 아래 두 메서드 호출과 동일합니다.

In [ ]:
# 물성 지도에서 군집 하나를 골라 좁히고
sizes = {c: n for c, n in root.placement.sizes().items() if c >= 0}
pick = sorted(sizes, key=lambda c: -sizes[c])[1]  # 가장 큰 것 말고 두 번째
n2 = session.select(root.node_id, clusters=[pick], note=f"물성 군집 {pick}")

# 같은 화합물을 질환 기준으로 재배치
n3 = session.switch_domain(n2.node_id, "disease", note="질환 관점")

print(session.breadcrumb(n3.node_id))
print(f"\n선택: {root.n:,} -> {n2.n:,} (영역 선택은 집합을 줄임)")
print(f"전환: {n2.n:,} -> {n3.n:,} (도메인 전환은 집합을 바꾸지 않음)")
print(f"군집: 물성 {len(n2.placement.cluster_ids)}개 -> 질환 {len(n3.placement.cluster_ids)}개")

### 관점이 실제로 다른지 확인

같은 화합물들의 물성 군집 라벨과 질환 군집 라벨이 얼마나 다른지 봅니다. 교차표가 대각선에서 멀수록 두 관점이 서로 독립이라는 뜻이고, 그게 이 도구를 쓰는 이유입니다.

In [ ]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score

phys, dis = n2.placement.labels, n3.placement.labels
both = (phys >= 0) & (dis >= 0)
print(f"ARI = {adjusted_rand_score(phys[both], dis[both]):.3f}  (0이면 완전히 독립)")
pd.crosstab(pd.Series(phys[both], name="물성 군집"), pd.Series(dis[both], name="질환 군집"))

## 계속: 질환에서 선택 → 표적으로

In [ ]:
sizes3 = n3.placement.sizes()
big3 = [c for c, _ in sorted(((c, n) for c, n in sizes3.items() if c >= 0), key=lambda kv: -kv[1])[:2]]
n4 = session.select(n3.node_id, clusters=big3, note=f"질환 군집 {big3}")
n5 = session.switch_domain(n4.node_id, "target", note="표적 관점")
print(session.breadcrumb(n5.node_id))

## 군집 프로파일

각 군집을 현재 도메인 + 물성 + 모달리티 관점에서 요약합니다.

In [ ]:
session.cluster_table(n5.node_id)

### 이 선택은 무엇인가

영역을 고른 직후 가장 자주 하게 되는 질문입니다. 지금 보고 있는 도메인뿐 아니라 다섯 도메인 전부에서 이 집합이 어떻게 보이는지, 그리고 각 도메인에 데이터가 얼마나 있는지를 함께 보여줍니다.

In [ ]:
session.describe(n5.node_id)

## 탐색 경로

트리이므로, 이전 노드로 돌아가 다른 영역을 선택하면 형제 가지가 추가됩니다. 이전 가지는 그대로 남아 비교할 수 있습니다.

In [ ]:
# n3으로 돌아가 다른 영역 선택 -> 형제 가지
others = [c for c in n3.placement.cluster_ids if c not in big3][:2]
if others:
    alt = session.select(n3.node_id, clusters=others, note="다른 질환 영역")
    session.switch_domain(alt.node_id, "modality", note="모달리티 관점")

print(session.tree(highlight=n5.node_id))

In [ ]:
session.history()

## 저장 / 복원

좌표와 군집까지 보존되므로, 복원한 세션은 저장 시점과 같은 지도를 보여줍니다.

In [ ]:
# data_path() 는 프로젝트 루트 기준으로 해석하므로, 노트북을 notebooks/ 에서
# 열어도 세션이 프로젝트의 data/ 아래에 저장됩니다.
path = session.save(chemspace.data_path("data/sessions/notebook"))
restored = chemspace.ExplorationSession.load(path, ds)
print(restored.tree())

## 화합물 단위로 꺼내기

어떤 노드든 좌표·군집이 붙은 데이터프레임으로 받을 수 있어서, 이후 분석(확보 후보 선별, 커버리지 비교 등)에 바로 연결됩니다.

In [ ]:
df = session.frame(n5.node_id)
df[["pref_name", "modality", "mw", "clogp", "murcko_smiles", "x", "y", "cluster"]].head(15)